# Clase 044 — SQL avanzado

**Parte 0** · Tanimura caps. 4-5.

> 🎯 CTEs (WITH), window functions (OVER), subqueries correlacionadas.

> ⏱️ ~120 min

## ⚙️ Setup — reutilizamos la BD de clase 041

In [ ]:
import sqlite3
import pandas as pd

con = sqlite3.connect(':memory:')
con.executescript('''
CREATE TABLE ordenes (
    orden_id INTEGER PRIMARY KEY,
    cliente_id INTEGER,
    fecha DATE,
    monto REAL
);
INSERT INTO ordenes (cliente_id, fecha, monto) VALUES
    (1, '2024-01-15', 120),(1, '2024-02-20',  80),(1, '2024-03-12', 150),(1, '2024-04-05',  60),
    (2, '2024-02-10',  40),(2, '2024-03-15',  70),(2, '2024-04-22',  50),
    (3, '2024-01-22', 200),(3, '2024-03-15', 180),(3, '2024-04-01', 220),
    (5, '2024-02-28', 300),(5, '2024-03-22',  90),(5, '2024-04-15', 400);
''')

## 1️⃣ CTEs (`WITH`) — descomponer queries

Una CTE es una "vista temporal" del scope de la query. Hace queries complejas legibles.

In [ ]:
# Sin CTE (anidado) — ilegible
q_nested = '''
SELECT pais, AVG(monto_cliente) AS avg_por_cliente
FROM (
    SELECT cliente_id, 'ES' AS pais, SUM(monto) AS monto_cliente
    FROM ordenes
    GROUP BY cliente_id
)
GROUP BY pais
'''

# Con CTE — paso a paso
q_cte = '''
WITH total_por_cliente AS (
    SELECT cliente_id, SUM(monto) AS monto
    FROM ordenes
    GROUP BY cliente_id
)
SELECT cliente_id, monto
FROM total_por_cliente
ORDER BY monto DESC
'''
print(pd.read_sql(q_cte, con))

## 2️⃣ Múltiples CTEs encadenadas

Pipeline legible: cada paso es una CTE con nombre:

In [ ]:
q = '''
WITH
monto_por_cliente AS (
    SELECT cliente_id, SUM(monto) AS total
    FROM ordenes
    GROUP BY cliente_id
),
ranking AS (
    SELECT cliente_id, total,
           ROW_NUMBER() OVER (ORDER BY total DESC) AS rnk
    FROM monto_por_cliente
)
SELECT * FROM ranking WHERE rnk <= 3
'''
print(pd.read_sql(q, con))

## 3️⃣ Window functions — el superpoder

```
FUNC() OVER (
    PARTITION BY grupo      -- (opcional) recalcula por cada grupo
    ORDER BY columna        -- (opcional) define el orden
    ROWS BETWEEN x AND y    -- (opcional) ventana móvil
)
```

No colapsa filas como `GROUP BY` — agrega información manteniendo cada fila.

In [ ]:
# Top-1 orden por cliente (mayor monto)
q = '''
WITH ranked AS (
    SELECT cliente_id, fecha, monto,
           ROW_NUMBER() OVER (PARTITION BY cliente_id ORDER BY monto DESC) AS rnk
    FROM ordenes
)
SELECT cliente_id, fecha, monto FROM ranked WHERE rnk = 1
'''
print('Top-1 orden por cliente:')
print(pd.read_sql(q, con))

## 4️⃣ Total corrido — `SUM() OVER (PARTITION BY ... ORDER BY ...)`

In [ ]:
q = '''
SELECT cliente_id, fecha, monto,
       SUM(monto) OVER (PARTITION BY cliente_id ORDER BY fecha) AS total_corrido
FROM ordenes
ORDER BY cliente_id, fecha
'''
print(pd.read_sql(q, con))

## 5️⃣ `LAG` / `LEAD` — comparar con fila anterior/siguiente

In [ ]:
q = '''
SELECT cliente_id, fecha, monto,
       LAG(monto)  OVER (PARTITION BY cliente_id ORDER BY fecha) AS monto_prev,
       monto - LAG(monto) OVER (PARTITION BY cliente_id ORDER BY fecha) AS delta
FROM ordenes
ORDER BY cliente_id, fecha
'''
print(pd.read_sql(q, con))

## 6️⃣ ROW_NUMBER vs RANK vs DENSE_RANK

```
valor: 10, 20, 20, 30
ROW_NUMBER: 1,  2,  3,  4    (siempre único)
RANK      : 1,  2,  2,  4    (huecos)
DENSE_RANK: 1,  2,  2,  3    (sin huecos)
```

## 7️⃣ Subqueries correlacionadas

Una subquery **correlacionada** se ejecuta una vez por cada fila de la outer query (referencia columnas del outer). Útiles pero suelen ser reescribibles con joins o window functions:

In [ ]:
# "clientes cuyo monto promedio supera el promedio global"
q = '''
SELECT cliente_id, AVG(monto) AS avg_cliente
FROM ordenes o
GROUP BY cliente_id
HAVING AVG(monto) > (SELECT AVG(monto) FROM ordenes)
'''
print(pd.read_sql(q, con))

## ✅ Checklist

- [ ] Uso CTEs para descomponer queries
- [ ] Encadeno múltiples CTEs
- [ ] Uso ROW_NUMBER OVER PARTITION para top-K por grupo
- [ ] Calculo totales corridos con SUM() OVER
- [ ] Comparo con anterior/siguiente con LAG/LEAD

## 📝 Homework

Ver `README.md`. 3 versiones (nested/CTE/multi-CTE), top-3 con ROW_NUMBER, total corrido + delta, recursive calendario.

## 📖 Definiciones y características

**CTE (Common Table Expression)**

Vista temporal dentro de una query con `WITH nombre AS (...)`. Descompone queries complejas en pasos legibles. Puedes encadenar múltiples: `WITH a AS (...), b AS (...) SELECT ...`.

**Recursive CTE**

CTE que se referencia a sí misma. Útil para jerarquías (árbol organizacional), grafos, generar series (calendario diario). Sintaxis: `WITH RECURSIVE t AS (caso_base UNION ALL caso_recursivo)`.

**Window function**

Agregación que **no colapsa filas** — añade el resultado por fila. Sintaxis: `FUNC() OVER (PARTITION BY col ORDER BY col2)`. Ejemplos: `ROW_NUMBER`, `RANK`, `LAG`, `LEAD`, `SUM() OVER (...)`.

**`PARTITION BY` vs `GROUP BY`**

**PARTITION BY** (en window): subgrupos para la función, pero mantiene cada fila. **GROUP BY**: reduce a una fila por grupo.

**`LAG` / `LEAD`**

Acceden a la fila anterior/siguiente dentro de la partition. `LAG(monto, 1) OVER (PARTITION BY cliente ORDER BY fecha)`. Útil para diffs, growth rates.

**Subquery correlacionada**

Subquery que **depende** de la outer query (referencia sus columnas). Se ejecuta una vez por cada fila de la outer. Más lenta que JOIN equivalente.

## ⚠️ Errores comunes

| Síntoma / mensaje | Causa y cómo arreglar |
|---|---|
| `syntax error at or near "OVER"` | Motor sin soporte de window functions (SQLite <3.25, MySQL <8). **Fix**: actualiza motor o reescribe con subqueries / self-join. |
| `ROW_NUMBER()` da números repetidos | Olvidaste `OVER (...)`. Sin él, no es window function. **Fix**: `ROW_NUMBER() OVER (ORDER BY col)`. |
| CTE recursiva nunca termina | Caso base falta o caso recursivo no converge. **Fix**: añade `LIMIT N` para debug, asegura que cada iteración acerca al caso base. |
| `LAG(x) OVER (ORDER BY fecha)` da NULL en la primera fila | Comportamiento esperado — no hay fila anterior. **Fix**: `LAG(x, 1, 0)` para default 0, o filtra con `WHERE row > 1`. |
| CTE da mismo resultado pero más lento que subquery | Algunos motores no inlineaban CTEs (PostgreSQL <12). **Fix**: actualiza, o reescribe como subquery temporalmente. |

## ❓ Preguntas frecuentes

**❓ ¿CTE o subquery?**

**CTE** si el lector necesita entender qué hace cada paso (legibilidad). **Subquery** si es trivial y de un solo uso. Para queries >10 líneas, CTE casi siempre gana.

**❓ ¿`ROW_NUMBER`, `RANK` o `DENSE_RANK`?**

Para valores `[10, 20, 20, 30]`: **ROW_NUMBER** `[1,2,3,4]` (siempre único). **RANK** `[1,2,2,4]` (huecos). **DENSE_RANK** `[1,2,2,3]` (sin huecos). Elige según semántica.

**❓ ¿Window function es lo mismo que groupby+merge en pandas?**

Conceptualmente sí — `g.transform(...)` en pandas hace lo equivalente. Window functions son la versión SQL más eficiente.

**❓ ¿Cuándo subquery correlacionada vs JOIN?**

Casi siempre **JOIN o window function** es más rápido. Correlacionada solo cuando no tiene equivalente JOIN (raro) o el optimizador del motor la maneja bien (motores modernos).

**❓ ¿Top-N por grupo?**

**Patrón estándar**: `WITH ranked AS (SELECT *, ROW_NUMBER() OVER (PARTITION BY grupo ORDER BY metric DESC) rn FROM tabla) SELECT * FROM ranked WHERE rn <= N`.

## 🔗 Referencias

- Tanimura, *SQL for Data Scientists*, caps. 4-5
- [Modern SQL — CTEs](https://modern-sql.com/feature/with)

➡️ **Siguiente:** [045 — SQL desde Python](../045-sql-desde-python-sqlite3-sqlalchemy-duckdb/README.md)

## ✅ Soluciones de los ejercicios

Soluciones trabajadas y **ejecutables sin internet** de los ejercicios del README. Cada bloque incluye comentarios y comprobaciones (`assert`/`print`). Intenta resolverlos tú antes de mirar.

In [ ]:
import sqlite3, pandas as pd
con = sqlite3.connect(':memory:')
con.executescript('''
CREATE TABLE clientes (id INTEGER PRIMARY KEY, nombre TEXT, pais TEXT);
CREATE TABLE ordenes  (id INTEGER PRIMARY KEY, cliente_id INTEGER, monto REAL, fecha TEXT);
''')
clientes = [(1,'Ana','ES'), (2,'Beto','CL'), (3,'Carla','ES'), (4,'Diego','MX'), (5,'Eva','ES')]
ordenes = [(1,1,120.0,'2024-01-05'), (2,1,50.0,'2024-01-09'), (3,1,90.0,'2024-02-01'),
           (4,1,30.0,'2024-02-10'), (5,2,200.0,'2024-01-12'),
           (6,3,300.0,'2024-01-15'), (7,3,150.0,'2024-01-20'), (8,3,60.0,'2024-02-02'),
           (9,3,45.0,'2024-02-11'), (10,4,80.0,'2024-01-30')]   # Eva (5) no tiene órdenes
con.executemany('INSERT INTO clientes VALUES (?,?,?)', clientes)
con.executemany('INSERT INTO ordenes  VALUES (?,?,?,?)', ordenes)
con.commit()
def q(sql):
    return pd.read_sql(sql, con)
print('clientes:', len(clientes), '| ordenes:', len(ordenes), '| sqlite', sqlite3.sqlite_version)

**Ejercicio 1 — CTE básica.** Reescribe una subquery anidada con `WITH`.

In [ ]:
r = q('''
WITH totales AS (
    SELECT cliente_id, SUM(monto) AS total
    FROM ordenes GROUP BY cliente_id
)
SELECT c.nombre, t.total
FROM totales t JOIN clientes c ON c.id = t.cliente_id
ORDER BY t.total DESC''')
print(r.to_string(index=False))
print('La CTE "totales" se define una vez y se reutiliza como si fuera una tabla.')

**Ejercicio 2 — ROW_NUMBER por grupo.** Top-1 orden por cliente (mayor monto).

In [ ]:
r = q('''
WITH ranked AS (
    SELECT cliente_id, monto, fecha,
           ROW_NUMBER() OVER (PARTITION BY cliente_id ORDER BY monto DESC) AS rn
    FROM ordenes)
SELECT c.nombre, r.monto AS mayor_monto, r.fecha
FROM ranked r JOIN clientes c ON c.id = r.cliente_id
WHERE r.rn = 1 ORDER BY r.cliente_id''')
print(r.to_string(index=False))
assert len(r) == 4   # 4 clientes con órdenes
print('rn = 1 selecciona, dentro de cada cliente, la orden de mayor monto')

**Ejercicio 3 — Total corrido.** `SUM(...) OVER (PARTITION BY cliente ORDER BY fecha)`.

In [ ]:
r = q('''
SELECT c.nombre, o.fecha, o.monto,
       SUM(o.monto) OVER (PARTITION BY o.cliente_id ORDER BY o.fecha) AS acumulado
FROM ordenes o JOIN clientes c ON c.id = o.cliente_id
ORDER BY o.cliente_id, o.fecha''')
print(r.to_string(index=False))
print('acumulado crece fila a fila DENTRO de cada cliente (running total).')

**Ejercicio 4 — LAG.** Diferencia entre el monto actual y el anterior por cliente.

In [ ]:
r = q('''
SELECT c.nombre, o.fecha, o.monto,
       o.monto - LAG(o.monto) OVER (PARTITION BY o.cliente_id ORDER BY o.fecha) AS dif_prev
FROM ordenes o JOIN clientes c ON c.id = o.cliente_id
ORDER BY o.cliente_id, o.fecha''')
print(r.to_string(index=False))
print('dif_prev = monto actual - anterior; es NULL en la 1ª orden de cada cliente.')

**Ejercicio 5 — Recursive CTE.** Serie de fechas día a día (2024-01-01 → 2024-01-31).

In [ ]:
r = q('''
WITH RECURSIVE dias(d) AS (
    SELECT DATE('2024-01-01')
    UNION ALL
    SELECT DATE(d, '+1 day') FROM dias WHERE d < DATE('2024-01-31'))
SELECT d FROM dias''')
assert len(r) == 31 and r.d.iloc[0] == '2024-01-01' and r.d.iloc[-1] == '2024-01-31'
print('Serie generada:', len(r), 'días, de', r.d.iloc[0], 'a', r.d.iloc[-1])